# Impact of Social Media on Life (ISML 2026)
## Behavioral Chronobiology, Psychometric Distress & Academic Performance Across 4,500 Students

> **ElmasCore Behavioral Intelligence Monograph** • High School to Postgraduate Multi-Cohort Analysis.

---

### Research Agenda
1. **Behavioral Cohort Census**: Granular mapping of 4,500 students across demographic, device, and digital consumption dimensions.
2. **Chronobiological Disruption**: Formulating and quantifying screen-to-sleep ratios, nocturnal friction, and cumulative weekly screen loads.
3. **Multi-Class Overall Impact**: Classifying students into `Beneficial`, `Neutral`, and `Negative` impact regimes with 5-fold stratified ensembling.
4. **Continuous Stress & GPA Modeling**: Forecasting perceived psychological distress (PSS 0-40) and academic GPA deterioration without target leakage.

In [1]:
import sys
from pathlib import Path
import json
import pandas as pd
import numpy as np
from IPython.display import Image, display

# Configure project root
PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.core.config import ProjectConfig
from src.data_pipeline.loader import load_raw_survey
from src.data_pipeline.schema_validator import validate_survey_schema
from src.feature_engineering.behavioral_features import compute_behavioral_features
from src.feature_engineering.encoder import build_feature_matrix

cfg = ProjectConfig()
print(f"ElmasCore ISML Pipeline Loaded. Project Root: {PROJECT_ROOT}")

## 1. Survey Ingestion & Cohort Baseline

We ingest the 4,500-student survey, apply deterministic median imputation on minor missing fields, and validate schema integrity.

In [2]:
raw_df = load_raw_survey(cfg.data.raw_data_path)
is_valid, errors = validate_survey_schema(raw_df)

print(f"Cohort Size: {len(raw_df):,} students")
print(f"Schema Valid: {is_valid} (Errors: {len(errors)})")
print(f"Age Span: {raw_df['Age'].min()} to {raw_df['Age'].max()} years (Mean: {raw_df['Age'].mean():.1f})")
print("\nOverall Impact Distribution:")
print(raw_df['Overall_Impact'].value_counts(normalize=True).round(4) * 100)

display(raw_df.head(5))

## 2. Chronobiological & Behavioral Feature Engineering

We calculate derived chronobiological and psychometric features:
- **Screen-to-Sleep Ratio**: $\text{Daily Screen Hours} / \text{Sleep Hours}$
- **Weekly Screen Load**: $5 \times \text{Weekday Hours} + 2 \times \text{Weekend Hours}$
- **Sleep Debt**: $\max(0, 8.0 - \text{Sleep Hours})$
- **Nocturnal Friction Index**: $\text{Late Night Flag} \times (6 - \text{Sleep Quality})$
- **Stress-to-Wellbeing Disparity**: $\text{PSS} / (\text{Mental Health} + 1)$

In [3]:
engineered_df = compute_behavioral_features(raw_df)
matrix = build_feature_matrix(engineered_df)

print(f"Engineered Features Count: {matrix.features.shape[1]}")
print(f"Feature Names: {matrix.feature_names}")

preview_cols = ['Daily_Usage_Hours', 'Sleep_Duration_Hours', 'screen_to_sleep_ratio', 'sleep_debt_hours', 'nocturnal_friction_index']
display(engineered_df[preview_cols].head(5))

## 3. Task 1: 3-Class Overall Impact Classification

5-Fold Stratified Soft-Voting Ensemble combining LightGBM and CatBoost with balanced class weighting.

In [4]:
with open(cfg.data.artifacts_dir / 'metrics_summary.json', 'r') as f:
    metrics_summary = json.load(f)

task1 = metrics_summary['overall_impact_classification']
print('=== Task 1: Overall Impact Classification (5-Fold CV) ===')
print(f"Overall Accuracy:      {task1['accuracy'] * 100:.2f}%")
print(f"Balanced Accuracy:     {task1['balanced_accuracy'] * 100:.2f}%")
print(f"Macro F1-Score:        {task1['macro_f1']:.4f}")
print(f"Weighted F1-Score:     {task1['weighted_f1']:.4f}")
print(f"Multi-Class Log-Loss:  {task1['multi_class_log_loss']:.4f}")
print('\nPer-Class F1 Scores:')
for k, v in task1['per_class_f1'].items():
    print(f"  - {k:12s}: {v:.4f}")

## 4. Task 2: Continuous Psychometric Distress Regression

Modeling the continuous continuum of psychological distress (PSS 0-40) with zero target leakage.

In [5]:
task2 = metrics_summary['psychometric_distress_regression']
print('=== Task 2: Continuous Stress Modeling ===')
print(f"RMSE:             {task2['rmse']:.4f}")
print(f"MAE:              {task2['mae']:.4f}")
print(f"R² Score:         {task2['r2_score']:.4f}")
print(f"Spearman Rho:     {task2['spearman_rho']:.4f} (p < 0.0001)")

## 5. Visual Artifacts & Empirical Telemetry Figures

Displaying the 5 publication-grade figures produced by the ElmasCore telemetry suite.

In [6]:
img_dir = cfg.data.images_dir
print('1. Behavioral Quadrant (Screen Time vs. Sleep Duration):')
display(Image(filename=str(img_dir / 'behavioral_clustering_quadrant.png')))
print('2. Normalized Confusion Matrix:')
display(Image(filename=str(img_dir / 'confusion_matrix_impact.png')))
print('3. Screen Time & Sleep Debt Tradeoff by Platform:')
display(Image(filename=str(img_dir / 'screen_time_sleep_debt_tradeoff.png')))
print('4. Dose-Response: Stress vs. Academic GPA:')
display(Image(filename=str(img_dir / 'gpa_stress_dose_response.png')))
print('5. Feature Importance & SHAP Attribution:')
display(Image(filename=str(img_dir / 'feature_importance_shap.png')))

## 6. Single-Student Triage CLI Demonstration

Querying student profiles for real-time risk assessment and chronobiological audit.

In [7]:
from predict import format_triage_report, triage_student

for sample_id in ['STU_2024000', 'STU_2024001', 'STU_2024002']:
    res = triage_student(sample_id, engineered_df)
    if res:
        print(format_triage_report(res))
        print()